# Module 1 — Brief 1 — Fine-tuning initial et évaluation

Ce notebook couvre les étapes du Brief 1 :

- Exploration du dataset FastIA
- Construction du prompt template
- Tokenisation et split train/test
- Chargement du modèle Llama 3.2 1B
- Configuration LoRA avec PEFT
- Entraînement avec tracking MLflow
- Evaluation quantitative et qualitative
- Diagnostic

Les cellules marquées `### A COMPLÉTER ###` contiennent du code partiel à terminer.
Les cellules sans marqueur sont fournies et peuvent être exécutées directement.

Le développement de l'API FastAPI (Brief 2) est hors de ce notebook.

---
## 0. Installation des dépendances

In [ ]:
# Exécuter cette cellule une seule fois
!pip install transformers peft datasets torch mlflow accelerate bitsandbytes

---
## 1. Imports et configuration

In [1]:
import json
import torch
import mlflow
from collections import Counter
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

print("Imports OK")

d:\Brice\Simplon_formation_ia\M1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [2]:
# Détection automatique du device disponible
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Device utilisé : {device}")

Device utilisé : cuda


---
## 2. Exploration du dataset

Avant d'entraîner quoi que ce soit, il est indispensable de comprendre les données.
Un modèle ne peut pas être meilleur que les données sur lesquelles il est entraîné.

Cette étape permet de détecter des déséquilibres de classes, des anomalies ou des
caractéristiques qui influenceront les choix d'entraînement.

In [3]:
# Chargement du dataset
with open("dataset_fastia_module1.jsonl", "r", encoding="utf-8") as f:
    raw_data = [json.loads(line) for line in f]

print(f"Nombre d'exemples : {len(raw_data)}")
print("\nExemple :")
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))

Nombre d'exemples : 86

Exemple :
{
  "input": "Bonjour, notre application de gestion RH plante depuis ce matin, impossible de se connecter. Toute l'équipe est bloquée.",
  "output": {
    "categorie": "Support technique",
    "priorite": "haute",
    "reponse_suggeree": "Nous prenons en charge votre incident en priorité. Notre équipe technique est mobilisée et reviendra vers vous dans l'heure avec un diagnostic."
  }
}


In [4]:
### A COMPLÉTER ###
# Objectif : analyser la distribution des catégories
#
# 1. Construire la liste de toutes les catégories présentes dans le dataset
# 2. Compter le nombre d'exemples par catégorie avec Counter
# 3. Afficher le résultat

categories = [exemple["output"]["categorie"] for exemple in raw_data]
distribution_categories = Counter(categories)

print("Distribution des catégories :")
for cat, count in distribution_categories.items():
    print(f"- {cat}: {count} exemples")

Distribution des catégories :
- Support technique: 20 exemples
- Demande commerciale: 16 exemples
- Demande de transformation: 15 exemples
- Réclamation: 15 exemples
- Information générale: 20 exemples


In [5]:
### A COMPLÉTER ###
# Objectif : analyser la distribution des priorités
#
# 1. Construire la liste de toutes les priorités
# 2. Compter le nombre d'exemples par priorité
# 3. Afficher le résultat

priorites = [ex['output']['priorite'] for ex in raw_data]
distribution_priorites = Counter(priorites)

print("Distribution des priorités :")
for priorite, count in distribution_priorites.items():
    print(f"- {priorite} : {count}")

Distribution des priorités :
- haute : 24
- normale : 62


In [6]:
### A COMPLÉTER ###
# Objectif : analyser la longueur des demandes en entrée
#
# 1. Calculer la longueur en caractères de chaque champ 'input'
# 2. Calculer la longueur minimale, maximale et moyenne
# 3. Afficher les résultats
#
# Question : pourquoi la longueur des inputs est-elle importante
# pour le choix de MAX_LENGTH lors de la tokenisation ?
# Répondez dans un commentaire.

longueurs = [len(ex['input']) for ex in raw_data]
longueur_minimale = min(longueurs)
longueur_maximale = max(longueurs)
longueur_moyenne = sum(longueurs) / len(longueurs)

print(f"Longueur minimale  : {longueur_minimale} caractères")
print(f"Longueur maximale  : {longueur_maximale} caractères")
print(f"Longueur moyenne   : {longueur_moyenne:.0f} caractères")

# Votre réponse :
#   Si MAX_LENGTH est défini trop petit, le modèle ne verra pas la fin des phrases présentent dans 'input',
#   perdant ainsi des informations essentielles pour la classification ou la réponse.
#   Si MAX_LENGTH est défini trop grand, cela peut entraîner une utilisation inefficace de la mémoire et du temps de calcul,
#   surtout si la majorité des inputs sont courts. Il est donc important de choisir un MAX_LENGTH qui couvre la majorité des inputs sans être excessivement grand.

Longueur minimale  : 44 caractères
Longueur maximale  : 140 caractères
Longueur moyenne   : 96 caractères


---
## 3. Construction du prompt template

Un modèle de langage apprend à partir de texte. Pour lui apprendre à classifier
et générer une réponse structurée en JSON, chaque exemple du dataset doit être
transformé en une paire instruction / réponse dans un format cohérent.

Ce format s'appelle un prompt template. Llama 3.2 utilise le format suivant :

```
<s>[INST] instruction [/INST] réponse </s>
```

Le modèle apprend à reproduire ce pattern : quand il voit [INST]...[/INST],
il génère la suite.

In [7]:
def build_prompt(exemple):
    """
    Transforme un exemple du dataset en prompt d'instruction.

    Args:
        exemple (dict): un exemple avec les clés 'input' et 'output'

    Returns:
        str: le prompt formaté
    """
    instruction = (
        "Tu es un assistant FastIA. "
        "Analyse la demande suivante et réponds uniquement en JSON valide "
        "avec les champs : categorie, priorite, reponse_suggeree.\n\n"
        f"Demande : {exemple['input']}"
    )
    reponse = json.dumps(exemple["output"], ensure_ascii=False)

    return f"<s>[INST] {instruction} [/INST] {reponse} </s>"


# Test sur le premier exemple
print(build_prompt(raw_data[0]))

<s>[INST] Tu es un assistant FastIA. Analyse la demande suivante et réponds uniquement en JSON valide avec les champs : categorie, priorite, reponse_suggeree.

Demande : Bonjour, notre application de gestion RH plante depuis ce matin, impossible de se connecter. Toute l'équipe est bloquée. [/INST] {"categorie": "Support technique", "priorite": "haute", "reponse_suggeree": "Nous prenons en charge votre incident en priorité. Notre équipe technique est mobilisée et reviendra vers vous dans l'heure avec un diagnostic."} </s>


In [8]:
### A COMPLÉTER ###
# Objectif : appliquer build_prompt à tous les exemples du dataset
#
# 1. Créer une liste 'prompts' contenant le prompt formaté de chaque exemple
# 2. Afficher la longueur de cette liste
# 3. Afficher le prompt du 5ème exemple pour vérification

prompts = [build_prompt(ex) for ex in raw_data]

print(f"Nombre de prompts : {len(prompts)}")
print("\nExemple (index 4) :")
print(prompts[4])

Nombre de prompts : 86

Exemple (index 4) :
<s>[INST] Tu es un assistant FastIA. Analyse la demande suivante et réponds uniquement en JSON valide avec les champs : categorie, priorite, reponse_suggeree.

Demande : URGENT : faille de sécurité détectée sur notre API exposée, des données clients pourraient être compromises. [/INST] {"categorie": "Support technique", "priorite": "haute", "reponse_suggeree": "Nous traitons cet incident de sécurité en priorité absolue. L'API sera isolée dans les plus brefs délais. Notre équipe sécurité vous contacte immédiatement."} </s>


---
## 4. Tokenisation et split train/test

Les modèles de langage ne travaillent pas avec du texte brut mais avec des tokens —
des unités de texte représentées par des entiers.

Le tokenizer est spécifique à chaque modèle. Il faut toujours utiliser le tokenizer
correspondant au modèle de base, jamais un tokenizer générique.

In [9]:
MODEL_ID = "meta-llama/Llama-3.2-1B"

# Chargement du tokenizer
# Un token d'accès HuggingFace est nécessaire pour ce modèle.
# Connectez-vous sur huggingface.co et acceptez les conditions d'utilisation Meta.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f"Taille du vocabulaire : {tokenizer.vocab_size} tokens")
print(f"Token de padding : {tokenizer.pad_token}")

Taille du vocabulaire : 128000 tokens
Token de padding : <|end_of_text|>


In [10]:
MAX_LENGTH = 512

def tokenize(prompt):
    """
    Tokenise un prompt et prépare les labels pour l'entraînement.

    Args:
        prompt (str): le prompt formaté

    Returns:
        dict: input_ids, attention_mask, labels
    """
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result


# Test sur un exemple
exemple_tokenise = tokenize(prompts[0])
print(f"Longueur tokenisée : {len(exemple_tokenise['input_ids'])} tokens")

Longueur tokenisée : 512 tokens


In [11]:
### A COMPLÉTER ###
# Objectif : construire les jeux d'entraînement et de test
#
# 1. Créer un Dataset HuggingFace à partir de la liste 'prompts'
#    Indice : Dataset.from_dict({"text": prompts})
#
# 2. Tokeniser tous les exemples avec .map()
#    Indice : dataset.map(lambda x: tokenize(x["text"]))
#
# 3. Diviser en train (80%) et test (20%) avec .train_test_split()
#
# 4. Afficher la taille des deux jeux
#
# Question : pourquoi conserver un jeu de test séparé du jeu d'entraînement ?
# Répondez dans un commentaire.

dataset = Dataset.from_dict({"text": prompts})
dataset_tokenise = dataset.map(lambda x: tokenize(x["text"]), batched=True)
split = dataset_tokenise.train_test_split(test_size=0.2)

train_dataset = split["train"]
test_dataset = split["test"]

print(f"Entraînement : {len(train_dataset)} exemples")
print(f"Test         : {len(test_dataset)} exemples")

# Votre réponse :
# Conserver un jeu de test séparé est indispensable pour :
# 1. Évaluer la généralisation : Vérifier si le modèle a réellement appris à traiter 
#    des demandes, ou s'il a simplement mémorisé (overfitting) les exemples de l'entrainement.
# 2. Détecter l'overfitting : Si la perte (loss) baisse sur l'entrainement mais stagne ou 
#    remonte sur le test, le modèle devient trop spécifique et perd en utilité réelle.
# 3. Mesurer la performance réelle : Obtenir des métriques fiables sur des données 
#    qu'il n'a jamais vues, simulant ainsi son usage en production.

Map: 100%|██████████| 86/86 [00:00<00:00, 1093.29 examples/s]

Entraînement : 68 exemples
Test         : 18 exemples


---
## 5. Chargement du modèle de base

On charge Llama 3.2 1B sans modifier ses poids. A ce stade, c'est le modèle
générique pré-entraîné par Meta sur des milliards de tokens.

L'étape suivante (LoRA) va ajouter des couches adaptateurs légères sur ce modèle
sans toucher aux poids originaux.

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Paramètres totaux : {total_params:,}")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:02<00:00, 58.63it/s]


Paramètres totaux : 1,235,814,400


---
## 6. Configuration LoRA

LoRA (Low-Rank Adaptation) est une technique de fine-tuning efficace : plutôt que
de modifier tous les poids du modèle, on ajoute de petites matrices d'adaptation
sur certaines couches.

Résultat : on entraîne seulement environ 1% des paramètres du modèle.

Paramètres clés :
- r : rang des matrices — plus élevé = plus de capacité, plus de mémoire
- lora_alpha : facteur d'échelle — généralement égal à 2 * r
- target_modules : couches sur lesquelles appliquer LoRA
- lora_dropout : régularisation pour éviter le sur-apprentissage

In [13]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


In [14]:
### A COMPLÉTER ###
# Objectif : calculer et afficher le pourcentage de paramètres entraînables
#
# 1. Calculer le nombre de paramètres entraînables (requires_grad == True)
# 2. Calculer le total des paramètres
# 3. Calculer et afficher le pourcentage
#
# Question : quel est l'avantage de n'entraîner qu'une fraction des paramètres
# sur un hardware avec une mémoire limitée ?
# Répondez dans un commentaire.

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = (trainable / total) * 100 if total > 0 else 0

print(f"Paramètres entraînables : {trainable:,} ({pct:.2f}% du total)")

# Votre réponse :
# L'avantage principal de n'entraîner qu'une fraction des paramètres (LoRA) est la réduction 
# drastique de la mémoire VRAM nécessaire. 
# En effet, lors de l'entraînement standard, on doit stocker les "états de l'optimiseur" 
# pour CHAQUE paramètre entraînable. 
# - Entraînement complet : Besoin en mémoire colossale car on stocke les gradients pour les milliards de paramètres.
# - LoRA : On ne stocke les gradients et les états de l'optimiseur que pour les 1% de paramètres ajoutés.
# Cela permet de fine-tuner des modèles massifs sur des cartes graphiques grand public)
# au lieu de serveurs industriels.

Paramètres entraînables : 851,968 (0.07% du total)


---
## 7. Entraînement

On lance le run d'entraînement. Chaque run est tracké dans MLflow pour conserver
une trace des paramètres et des métriques.

Hyperparamètres à retenir :
- learning_rate : vitesse d'apprentissage
- num_train_epochs : nombre de passages sur le dataset complet
- per_device_train_batch_size : nombre d'exemples traités simultanément

In [ ]:
mlflow.set_experiment("fastia-llama-finetuning")

TRAINING_CONFIG = {
    "learning_rate": 2e-4,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "run_name": "run1-lr2e4-3epochs"
}

with mlflow.start_run(run_name=TRAINING_CONFIG["run_name"]):

    mlflow.log_params({
        "model_id": MODEL_ID,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "learning_rate": TRAINING_CONFIG["learning_rate"],
        "num_train_epochs": TRAINING_CONFIG["num_train_epochs"],
        "batch_size": TRAINING_CONFIG["per_device_train_batch_size"],
        "dataset_size": len(raw_data),
        "max_length": MAX_LENGTH
    })

    training_args = TrainingArguments(
        output_dir="./model_run1",
        num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
        per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
        learning_rate=TRAINING_CONFIG["learning_rate"],
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        fp16=torch.cuda.is_available(),
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        processing_class=tokenizer
    )

    result = trainer.train()

    mlflow.log_metric("train_loss", result.training_loss)

    # Retirer le callback lié au Notebook ---
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)

    eval_result = trainer.evaluate()
    mlflow.log_metric("eval_loss", eval_result["eval_loss"])

    print(f"Train loss : {result.training_loss:.4f}")
    print(f"Eval loss  : {eval_result['eval_loss']:.4f}")

2026/04/14 11:35:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/14 11:35:27 INFO mlflow.store.db.utils: Updating database tables
2026/04/14 11:35:29 INFO mlflow.tracking.fluent: Experiment with name 'fastia-llama-finetuning' does not exist. Creating a new experiment.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Epoch,Training Loss,Validation Loss
1,5.876563,0.843535
2,0.760456,0.675093
3,0.596403,0.593054


Train loss : 1.7566
Eval loss  : 0.5931


---
## 8. Evaluation quantitative

La loss mesure l'erreur du modèle sur les données. Elle ne dit pas directement
si les prédictions sont correctes, mais elle indique si le modèle apprend.

Deux signaux importants à analyser :
- La courbe de loss : descend-elle régulièrement ?
- L'écart train loss / eval loss : un écart important signale un sur-apprentissage.

In [ ]:
### A COMPLÉTER ###
# Objectif : récupérer et afficher les métriques du run depuis MLflow
#
# 1. Récupérer les runs de l'expérience avec mlflow.search_runs()
#    Indice : mlflow.search_runs(experiment_names=["fastia-llama-finetuning"])
#
# 2. Afficher les colonnes suivantes pour le run :
#    - run_name
#    - learning_rate
#    - num_train_epochs
#    - train_loss
#    - eval_loss
#
# Question : que vous indique l'écart entre train_loss et eval_loss ?
# Répondez dans un commentaire.

runs = mlflow.search_runs(experiment_names=["fastia-llama-finetuning"])

colonnes_interessantes = [
    "tags.mlflow.runName",
    "params.learning_rate",
    "params.num_train_epochs",
    "metrics.train_loss",
    "metrics.eval_loss"
]

print(runs[colonnes_interessantes])

# Votre réponse :
# les données : 
# Epoch	Training Loss	Validation Loss
#   1	    5.876563	    0.843535
#   2	    0.760456	    0.675093
#   3	    0.596403	    0.593054
#    tags.mlflow.runName   params.learning_rate    params.num_train_epochs metrics.train_loss  metrics.eval_loss
# 0  run1-lr2e4-3epochs               0.0002                       3              1.756628           0.593054
#    
# Au court des 3 epochs, on observe une diminution significative du train_loss, passant de 5.876 à 0.596, 
# ce qui indique que le modèle apprend à mieux s'ajuster aux données d'entraînement.
# À la fin de la 3ème époque, le train_loss est proche de l'eval_loss, ce qui suggère que le modèle généralise plutôt bien.
# L'ecart entre trains_loss et eval_loss est de : 1.7566 − 0.5930 = 1.1636
# cet écart est important, cependant il est a noté que le train_loss a beaucoup diminué au cours des epochs, ce qui est un signe positif d'apprentissage.

  tags.mlflow.runName params.learning_rate params.num_train_epochs  \
0  run1-lr2e4-3epochs               0.0002                       3   

   metrics.train_loss  metrics.eval_loss  
0            1.756628           0.593054  


---
## 9. Evaluation qualitative

La loss est une métrique indirecte. Il faut aussi vérifier que le modèle produit
des sorties réellement utilisables : JSON valide, catégories reconnues, réponses
cohérentes avec la demande.

In [58]:
def predict(texte, model, tokenizer, max_new_tokens=150):
    """
    Génère une prédiction à partir d'un texte de demande.

    Args:
        texte (str): la demande client en texte libre
        model: le modèle fine-tuné
        tokenizer: le tokenizer associé
        max_new_tokens (int): nombre maximum de tokens à générer

    Returns:
        dict: la réponse parsée en JSON, ou un dict d'erreur si le parsing échoue
    """
    prompt = (
        f"<s>[INST] Tu es un assistant FastIA. "
        f"Analyse la demande suivante et réponds uniquement en JSON valide "
        f"avec les champs : categorie, priorite, reponse_suggeree.\n\n"
        f"Demande : {texte} [/INST]"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

# --- AJOUT : Nettoyage de la sortie ---
    # 1. On retire le token de fin de séquence résiduel
    generated = generated.replace("</s>", "")
    # 2. On retire les éventuelles balises Markdown
    generated = generated.replace("```json", "").replace("```", "")
    # 3. On enlève les espaces ou sauts de ligne restants
    generated = generated.strip()
        
    try:
        return json.loads(generated)
    except json.JSONDecodeError:
        return {"erreur": "JSON invalide", "brut": generated}


# Test rapide
test_input = "Notre serveur est tombé, personne ne peut travailler."
print(json.dumps(predict(test_input, model, tokenizer), ensure_ascii=False, indent=2))

{
  "categorie": "urgence",
  "priorite": "1",
  "reponse_suggeree": "Nous avons un serveur en panne, il faudra attendre que l'incident soit résolu."
}


In [ ]:
### A COMPLÉTER ###
# Objectif : évaluer le modèle sur 10 demandes couvrant les 5 catégories
#
# 1. Choisir 10 demandes (au moins une par catégorie)
#    Les demandes ne doivent pas être copiées du dataset d'entraînement
#
# 2. Pour chaque demande, appeler predict() et vérifier :
#    - Le JSON est-il valide ? (pas de clé 'erreur' dans le résultat)
#    - La catégorie est-elle dans CATEGORIES_VALIDES ?
#    - La priorité est-elle 'haute' ou 'normale' ?
#
# 3. Afficher un tableau récapitulatif :
#    Demande | Catégorie prédite | Priorité | JSON valide

CATEGORIES_VALIDES = [
    "Support technique",
    "Demande commerciale",
    "Demande de transformation",
    "Réclamation",\
    "Information générale"
]

demandes_test = [
    # A COMPLÉTER : 10 demandes de votre choix
    "Mon ordinateur ne s'allume plus et fait un bruit bizarre.", # Support technique
    "J'aimerais obtenir un devis pour l'achat de 10 nouvelles licences.", # Demande commerciale
    "Pouvez-vous migrer mon compte actuel vers l'offre Entreprise ?", # Demande de transformation
    "Je suis très mécontent, mon colis est arrivé cassé !", # Réclamation
    "Quels sont vos tarifs pour le support premium ?", # Information générale
    "Le logiciel plante dès que je clique sur le bouton imprimer.", # Support technique
    "Je souhaite résilier mon abonnement et passer chez un concurrent.", # Réclamation
    "Est-il possible d'ajouter une option stockage cloud à mon contrat ?", # Demande de transformation
    "Je voudrais parler à un commercial pour un partenariat.", # Demande commerciale
    "Avez-vous une documentation sur l'utilisation de l'API ?" # Information générale
]

resultats = []

print("Analyse des demandes en cours...")

for demande in demandes_test:
    prediction = predict(demande, model, tokenizer)
    
    # Initialisation des variables de contrôle
    categorie_predite = "N/A"
    priorite_predite = "N/A"
    json_valide = "Non"

    if isinstance(prediction, dict) and "erreur" not in prediction:
        json_valide = "Oui"
        categorie_predite = prediction.get("categorie", "N/A")
        priorite_predite = prediction.get("priorite", "N/A")

        # Validation des valeurs
        cat_ok = categorie_predite in CATEGORIES_VALIDES
        prio_ok = priorite_predite in ["haute", "normale"]
        
        if not (cat_ok and prio_ok):
            json_valide = "Partiel (Valeurs invalides)"
    else:
            json_valide = "Non (JSON Invalide)"
    
    # 3. Stockage pour le tableau
    resultats.append({
        "Demande": demande[:40] + "...",
        "Catégorie prédite": categorie_predite,
        "Priorité": priorite_predite,
        "JSON valide": json_valide
    })

# 3. Affichage du tableau récapitulatif
import pandas as pd
df_res = pd.DataFrame(resultats)

print("\n--- TABLEAU RÉCAPITULATIF DES PERFORMANCES ---")
print(df_res.to_string(index=False))

Analyse des demandes en cours...

--- TABLEAU RÉCAPITULATIF DES PERFORMANCES ---
                                    Demande   Catégorie prédite Priorité                 JSON valide
Mon ordinateur ne s'allume plus et fait ...           technique        1 Partiel (Valeurs invalides)
J'aimerais obtenir un devis pour l'achat...            Licences     Haut Partiel (Valeurs invalides)
Pouvez-vous migrer mon compte actuel ver...             Demande  Moyenne Partiel (Valeurs invalides)
Je suis très mécontent, mon colis est ar...                 N/A      N/A         Non (JSON Invalide)
Quels sont vos tarifs pour le support pr...             support     high Partiel (Valeurs invalides)
Le logiciel plante dès que je clique sur...             Demande  Moyenne Partiel (Valeurs invalides)
Je souhaite résilier mon abonnement et p...          abonnement     high Partiel (Valeurs invalides)
Est-il possible d'ajouter une option sto...             contrat     high Partiel (Valeurs invalides)
Je voudrai

---
## 10. Diagnostic

Le diagnostic est une étape de réflexion structurée sur les résultats obtenus.
Il conditionne les actions correctives du Brief 2.

Répondez aux questions suivantes dans les cellules ci-dessous.
Appuyez-vous sur les métriques MLflow et sur les résultats qualitatifs.

In [ ]:
### A COMPLÉTER ###
# Question 1 : classification
#
# Les données :
# --- TABLEAU RÉCAPITULATIF DES PERFORMANCES ---
#                                     Demande   Catégorie prédite Priorité                 JSON valide
# Mon ordinateur ne s'allume plus et fait ...           technique        1 Partiel (Valeurs invalides)
# J'aimerais obtenir un devis pour l'achat...            Licences     Haut Partiel (Valeurs invalides)
# Pouvez-vous migrer mon compte actuel ver...             Demande  Moyenne Partiel (Valeurs invalides)
# Je suis très mécontent, mon colis est ar...                 N/A      N/A         Non (JSON Invalide)
# Quels sont vos tarifs pour le support pr...             support     high Partiel (Valeurs invalides)
# Le logiciel plante dès que je clique sur...             Demande  Moyenne Partiel (Valeurs invalides)
# Je souhaite résilier mon abonnement et p...          abonnement     high Partiel (Valeurs invalides)
# Est-il possible d'ajouter une option sto...             contrat     high Partiel (Valeurs invalides)
# Je voudrais parler à un commercial pour ... Demande commerciale  Moyenne Partiel (Valeurs invalides)
# Avez-vous une documentation sur l'utilis...                 API  Moyenne Partiel (Valeurs invalides)

# Le modèle classifie-t-il correctement toutes les catégories ?
#   Non, le modèle ne classifie pas correctement, de plus il ne respecte pas les noms des catégories, ni des priorités.

# Certaines catégories sont-elles mieux gérées que d'autres ?
#   Une "Demande commerciale" à bien été gérée, les autres catégories présentent des erreurs de classification ou de formatage, 
#
# Les erreurs sont-elles aléatoires ou systématiques ?
#   Les erreurs sont systématiques : le modèle souffre de comportements répétitifs.
#   Logique de "Mot-Clé" (Extraction vs Classification) : Le modèle n'utilise pas vos 5 catégories prédéfinies. 
#   Il extrait systématiquement un mot important du texte (ex: "API", "Abonnement", "Licences") pour en faire une étiquette. 
#
#   Problème de format JSON: L'échec du JSON sur 100% des tests montre que le modèle n'a pas intégré la contrainte du format JSON comme une règle absolue. 
#   Conflit de Langue: dans les prioritées, L'alternance entre le français ("Moyenne") et l'anglais ("high") montre que le modèle n'a pas intégré 
#   les contraintes de noms.


In [ ]:
### A COMPLÉTER ###
# Question 2 : lecture de la loss
#
# les données : 
# Epoch	Training Loss	Validation Loss
#   1	    5.876563	    0.843535
#   2	    0.760456	    0.675093
#   3	    0.596403	    0.593054
#    tags.mlflow.runName   params.learning_rate    params.num_train_epochs metrics.train_loss  metrics.eval_loss
# 0  run1-lr2e4-3epochs               0.0002                       3              1.756628           0.593054
#    
# Que vous indique l'écart entre train_loss et eval_loss ?
#   Au court des 3 epochs, on observe une diminution significative du train_loss, passant de 5.876 à 0.596, 
#   ce qui indique que le modèle apprend à mieux s'ajuster aux données d'entraînement.
#   À la fin de la 3ème époque, le train_loss est proche de l'eval_loss, ce qui suggère que le modèle généralise plutôt bien.
#   L'ecart entre trains_loss et eval_loss est de : 1.7566 − 0.5930 = 1.1636
#   cet écart est important, cependant il est a noté que le train_loss a beaucoup diminué au cours des epochs, ce qui est un signe positif d'apprentissage.

# Le modèle est-il en sous-apprentissage ou en sur-apprentissage ?
#   Le modèle semble en sous-apprentissage (underfitting).

# Justifiez à partir des valeurs observées.
#   Le modèle a appris à prédire du texte cohérent, mais il sous-apprend les tâches spécifiques :
#   Échec des contraintes : Il n'a pas assimilé les règles rigides du format JSON ni la liste fermée des catégories et des priorités,
#   le modèle répond parfois en anglais


In [ ]:
### A COMPLÉTER ###
# Question 3 : qualité de la sortie JSON
#
# Le modèle génère-t-il systématiquement du JSON valide ?
#   Non, le modèle ne génère pas de JSON valide

# Si non, dans quels cas le JSON est-il invalide ou mal formé ?
#   dans chaque cas testé, le JSON est invalide


In [ ]:
### A COMPLÉTER ###
# Question 4 : actions correctives
#
# Sur la base de votre diagnostic, proposez les actions correctives
# que vous mettrez en oeuvre dans le Brief 2.
#
# Pour chaque action, précisez :
# - Ce que vous allez faire (compléter le dataset, ajuster les hyperparamètres, les deux)
# - Pourquoi cette action devrait améliorer les résultats
# - Ce que vous espérez observer sur la loss et les prédictions

# Vos propositions :
# - Ce que vous allez faire (compléter le dataset, ajuster les hyperparamètres, les deux)
#   Agir sur les deux leviers.Compléter le dataset en ajoutant des exemples pour les catégories sous-représentées.
#   Ajuster les hyperparamètres en passant de 3 à 4 époques 
#   et Reformuler le "System Prompt" pour insister sur la liste stricte des catégories et priorités autorisées.

# - Pourquoi cette action devrait améliorer les résultats
#   Le nouveau "System Prompt" va imposer un cadre strict (nomenclature et JSON).
#   L'ajout d'exemples permettra au modèl de mieux apprendre à généraliser les cas complexes.
#   Le passage à 4 époques permettra au modèle une meilleure compréhension globale du format attendu et des catégories et des priorités.
#
# - Ce que vous espérez observer sur la loss et les prédictions
#   Sur la loss : Je m'attends à ce que la train_loss descende de manière régulière et se stabilise à un niveau bas, 
#   proche de la eval_loss.
#   Sur les prédictions : un respect du format JSON, des catégories conformes à la liste prédéfinie, et des priorités correctes."

---
## Récapitulatif

Etapes réalisées dans ce notebook :

1. Exploration du dataset : distribution des catégories, priorités, longueurs
2. Construction du prompt template d'instruction
3. Tokenisation et split train/test
4. Chargement de Llama 3.2 1B
5. Configuration et application de LoRA
6. Entraînement tracké dans MLflow
7. Evaluation quantitative : analyse de la loss
8. Evaluation qualitative : 10 prédictions sur cas réels
9. Diagnostic et propositions d'actions correctives

Prochaine étape — Brief 2 : mettre en oeuvre les actions correctives,
réentraîner, comparer les runs et exposer le meilleur modèle via FastAPI.